# Day 3 — Gold Layer: Aggregations & Analytics
### PySpark groupBy · Window functions · SQL RANK / LAG / anti-join / cohort
### Retail Analytics DE Project

---

> **Project Day:** 3 · **Layer:** Gold  
> **Input:** `silver.*` tables in PostgreSQL  
> **Output:** `gold.*` tables in PostgreSQL  
> **Prerequisite:** Day 2 complete — silver tables populated

| Section | Topic |
|---------|-------|
| 1 | Setup — import config + Spark |
| 2 | Gold — Daily Sales Summary |
| 3 | Gold — Monthly Revenue by State |
| 4 | Gold — RFM Customer Segmentation |
| 5 | Gold — Product Revenue with cumulative window |
| 6 | Gold — SQL Analytics (RANK, LAG, anti-join, cohort) |
| 7 | Gold Summary |

---
## 1. Setup — Import config + Spark

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from config.db_config import (
    get_engine, set_spark_env, get_spark,
    spark_read_pg, spark_write_pg
)

engine = get_engine()
print('Engine ready:', engine.url)

In [ ]:
set_spark_env()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, DoubleType

spark = get_spark('Day3_Gold')
print('Spark ready:', spark.version)

In [ ]:
from sqlalchemy import text, inspect as sa_inspect
from datetime import date, datetime

GOLD_AT = datetime.utcnow().isoformat()

def read_silver(table):
    """Read a silver table into a PySpark DataFrame via JDBC."""
    df = spark_read_pg(spark, 'silver', table)
    print(f'  silver.{table}: {df.count()} rows')
    return df

def to_gold(df, table):
    """Add _gold_loaded_at and write to gold schema via JDBC."""
    df = df.withColumn('_gold_loaded_at', F.lit(GOLD_AT))
    spark_write_pg(df, 'gold', table, mode='overwrite')

print('Helpers ready. GOLD_AT =', GOLD_AT)

---
---
## 2. Gold — Daily Sales Summary

---

Aggregate non-cancelled orders by calendar day.  
Metrics: `total_revenue`, `order_count`, `unique_customers`, `avg_order_value`.

In [ ]:
df_orders = read_silver('orders')
df_orders.printSchema()

In [ ]:
# Exclude cancelled orders
df_active = df_orders.filter(F.col('is_cancelled') == False)

# Cast total_amount to double (may be object type after roundtrip through pandas)
df_active = df_active.withColumn('total_amount', F.col('total_amount').cast(DoubleType()))

# Group by calendar day
df_daily = (
    df_active
    .withColumn('order_day', F.to_date(F.col('order_date')))
    .groupBy('order_day')
    .agg(
        F.round(F.sum('total_amount'),   2).alias('total_revenue'),
        F.count('order_id')               .alias('order_count'),
        F.countDistinct('customer_id')    .alias('unique_customers'),
    )
    .withColumn('avg_order_value', F.round(F.col('total_revenue') / F.col('order_count'), 2))
    .orderBy('order_day')
)

print(f'Daily rows: {df_daily.count()}')
print('Top 5 days by revenue:')
df_daily.orderBy(F.col('total_revenue').desc()).show(5, truncate=False)
to_gold(df_daily, 'daily_sales')

---
---
## 3. Gold — Monthly Revenue by State

---

In [ ]:
df_customers = read_silver('customers')

# Join orders with customer state
df_monthly = (
    df_active
    .join(df_customers.select('customer_id', 'state'), on='customer_id', how='left')
    .withColumn('order_month', F.date_format(F.col('order_date'), 'yyyy-MM'))
    .groupBy('order_month', 'state')
    .agg(
        F.round(F.sum('total_amount'), 2).alias('total_revenue'),
        F.count('order_id')             .alias('order_count'),
    )
    .orderBy('order_month', F.col('total_revenue').desc())
)

print(f'Monthly-state rows: {df_monthly.count()}')
df_monthly.show(10, truncate=False)
to_gold(df_monthly, 'monthly_sales_by_state')

---
---
## 4. Gold — RFM Customer Segmentation

---

**RFM stands for:**
- **R** Recency — days since last order (lower = better)
- **F** Frequency — number of orders (higher = better)
- **M** Monetary — total spend (higher = better)

**PySpark approach:**
1. Compute raw RFM per customer with `groupBy + agg`
2. Score 1–4 with `ntile(4)` window function (quartiles)
3. Derive segment with nested `WHEN` expressions

In [ ]:
# Step 1: compute raw RFM per customer
reference_date = F.lit(str(date.today()))

rfm_raw = (
    df_active
    .withColumn('order_date_d', F.to_date(F.col('order_date')))
    .groupBy('customer_id')
    .agg(
        F.datediff(F.lit(str(date.today())), F.max('order_date_d')).alias('recency_days'),
        F.count('order_id')                                        .alias('frequency'),
        F.round(F.sum('total_amount'), 2)                          .alias('monetary'),
    )
)

print('Raw RFM (top 5 by monetary):')
rfm_raw.orderBy(F.col('monetary').desc()).show(5, truncate=False)

In [ ]:
# Step 2: score with ntile(4) window (quartiles — no ties issue)
# R: rank ascending (lowest recency = most recent = score 4)
w_r = Window.orderBy('recency_days')   # ascending → ntile 1=most recent
w_f = Window.orderBy('frequency')      # ascending → ntile 4=most frequent
w_m = Window.orderBy('monetary')       # ascending → ntile 4=highest spend

rfm_scored = (
    rfm_raw
    # R: flip so 4=best (most recent). ntile 1 = lowest recency → best
    .withColumn('r_raw', F.ntile(4).over(w_r))
    .withColumn('r_score', (5 - F.col('r_raw')).cast(IntegerType()))  # invert: 1→4, 4→1
    .withColumn('f_score', F.ntile(4).over(w_f).cast(IntegerType()))
    .withColumn('m_score', F.ntile(4).over(w_m).cast(IntegerType()))
    .withColumn('rfm_score',
        F.col('r_score') + F.col('f_score') + F.col('m_score')
    )
    .drop('r_raw')
)

print('Scored RFM:')
rfm_scored.select('customer_id','recency_days','frequency','monetary','r_score','f_score','m_score','rfm_score') \
    .orderBy(F.col('rfm_score').desc()).show(8, truncate=False)

In [ ]:
# Step 3: segment with WHEN (mirrors the pandas assign_segment function)
rfm_final = (
    rfm_scored
    .withColumn('segment',
        F.when((F.col('r_score') >= 4) & (F.col('f_score') >= 3) & (F.col('m_score') >= 3), F.lit('Champion'))
         .when((F.col('f_score') >= 3) & (F.col('m_score') >= 3), F.lit('Loyal'))
         .when((F.col('r_score') <= 2) & (F.col('f_score') >= 2), F.lit('At Risk'))
         .otherwise(F.lit('Lost'))
    )
    # Enrich with customer name
    .join(df_customers.select('customer_id','full_name','city','state'), on='customer_id', how='left')
)

print('Segment distribution:')
rfm_final.groupBy('segment').count().orderBy(F.col('count').desc()).show()

print('Champions:')
rfm_final.filter(F.col('segment') == 'Champion') \
    .select('customer_id','full_name','recency_days','frequency','monetary','rfm_score','segment') \
    .show(truncate=False)

to_gold(rfm_final, 'customer_rfm_segments')

---
---
## 5. Gold — Product Revenue with Cumulative Window

---

**Cumulative window:** `SUM(monthly_revenue) OVER (PARTITION BY product_id ORDER BY order_month ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)`  
Shows how revenue for each product has compounded over time.

In [ ]:
df_items = read_silver('order_items')

df_items = df_items.withColumn('line_total', F.col('line_total').cast(DoubleType()))

# Join with orders to get order_date
df_ord_dates = df_orders.select('order_id', 'order_date', 'is_cancelled') \
    .filter(F.col('is_cancelled') == False)

df_joined = df_items.join(df_ord_dates, on='order_id', how='inner')

In [ ]:
# Total revenue per product
df_prod_total = (
    df_joined
    .groupBy('product_id')
    .agg(
        F.round(F.sum('line_total'), 2) .alias('total_revenue'),
        F.sum('quantity')               .alias('units_sold'),
        F.countDistinct('order_id')     .alias('order_count'),
        F.round(F.avg('line_total'), 2) .alias('avg_line_total'),
    )
    .orderBy(F.col('total_revenue').desc())
)
print('Top 10 products by revenue:')
df_prod_total.show(10, truncate=False)
to_gold(df_prod_total, 'product_revenue')

In [ ]:
# Monthly revenue per product + cumulative window
df_prod_monthly = (
    df_joined
    .withColumn('order_month', F.date_format(F.col('order_date'), 'yyyy-MM'))
    .groupBy('product_id', 'order_month')
    .agg(
        F.round(F.sum('line_total'), 2).alias('monthly_revenue'),
        F.sum('quantity')              .alias('units_sold'),
    )
)

# Cumulative window within each product, ordered by month
w_cum = Window.partitionBy('product_id').orderBy('order_month').rowsBetween(
    Window.unboundedPreceding, Window.currentRow
)
df_prod_monthly = df_prod_monthly.withColumn(
    'cumulative_revenue', F.round(F.sum('monthly_revenue').over(w_cum), 2)
)

print('Monthly + cumulative for top product:')
top_prod = df_prod_total.first()['product_id']
df_prod_monthly.filter(F.col('product_id') == top_prod).orderBy('order_month').show(truncate=False)

to_gold(df_prod_monthly, 'product_revenue_monthly')

---
---
## 6. Gold — SQL Analytics

---

| Query | SQL Feature |
|-------|------------|
| Top customers by revenue | `RANK() OVER (ORDER BY revenue DESC)` |
| Month-over-month growth | `LAG(revenue) OVER (ORDER BY month)` |
| Products never ordered | `LEFT JOIN … WHERE right.id IS NULL` (anti-join) |
| Customer acquisition cohort | `SUM() OVER (ORDER BY month)` cumulative |

In [ ]:
from sqlalchemy import text
from config.db_config import JDBC_URL, DB_USER, DB_PASS

# ── Top customers with RANK ────────────────────────────────────────────────────
top_cust_sql = """
CREATE TABLE IF NOT EXISTS gold.top_customers AS
WITH customer_revenue AS (
    SELECT
        o.customer_id,
        c.full_name,
        c.state,
        ROUND(SUM(o.total_amount)::NUMERIC, 2)  AS total_revenue,
        COUNT(DISTINCT o.order_id)               AS order_count
    FROM silver.orders o
    JOIN silver.customers c ON o.customer_id = c.customer_id
    WHERE o.is_cancelled = FALSE
    GROUP BY o.customer_id, c.full_name, c.state
)
SELECT *,
    RANK() OVER (ORDER BY total_revenue DESC) AS revenue_rank
FROM customer_revenue;
"""

with engine.connect() as conn:
    conn.execute(text('DROP TABLE IF EXISTS gold.top_customers;'))
    conn.execute(text(top_cust_sql))
    conn.commit()

print('Top customers (RANK):')
df_top = spark_read_pg(spark, 'gold', 'top_customers')
df_top.orderBy('revenue_rank').show(10, truncate=False)

In [ ]:
# ── Month-over-month growth with LAG ──────────────────────────────────────────
# Create a view for the MoM query so we can read it via JDBC
mom_view_sql = """
CREATE OR REPLACE VIEW gold.v_mom_growth AS
WITH monthly AS (
    SELECT DATE_TRUNC('month', order_date) AS order_month,
           ROUND(SUM(total_amount)::NUMERIC, 2) AS revenue
    FROM silver.orders
    WHERE is_cancelled = FALSE
    GROUP BY DATE_TRUNC('month', order_date)
)
SELECT
    TO_CHAR(order_month, 'YYYY-MM') AS month,
    revenue,
    ROUND(LAG(revenue) OVER (ORDER BY order_month)::NUMERIC, 2) AS prev_revenue,
    CASE WHEN LAG(revenue) OVER (ORDER BY order_month) IS NOT NULL
         THEN ROUND(100.0*(revenue - LAG(revenue) OVER (ORDER BY order_month))
                    / LAG(revenue) OVER (ORDER BY order_month), 1)
    END AS growth_pct
FROM monthly
ORDER BY order_month;
"""
with engine.connect() as conn:
    conn.execute(text(mom_view_sql))
    conn.commit()

print('Month-over-month growth (LAG):')
spark_read_pg(spark, 'gold', 'v_mom_growth').show(truncate=False)

In [ ]:
# ── Anti-join: products never ordered ─────────────────────────────────────────
# PySpark left_anti join — no SQL needed
df_products_all = spark_read_pg(spark, 'silver', 'products')
df_ordered_prods = spark_read_pg(spark, 'silver', 'order_items').select('product_id').distinct()

df_never_ordered = (
    df_products_all
    .join(df_ordered_prods, on='product_id', how='left_anti')
    .select('product_id', 'product_name', 'category', 'unit_price')
    .orderBy('category', 'product_name')
)

print('Products never ordered (PySpark left_anti join):')
df_never_ordered.show(truncate=False)

In [ ]:
# ── Category revenue table ─────────────────────────────────────────────────────
cat_sql = """
CREATE TABLE IF NOT EXISTS gold.category_revenue AS
SELECT
    p.category,
    COUNT(DISTINCT oi.order_id)                                        AS orders,
    SUM(oi.quantity)                                                   AS units_sold,
    ROUND(SUM(oi.line_total)::NUMERIC, 2)                              AS revenue,
    ROUND(100.0 * SUM(oi.line_total) / SUM(SUM(oi.line_total)) OVER (), 1) AS share_pct,
    RANK() OVER (ORDER BY SUM(oi.line_total) DESC)                     AS revenue_rank
FROM silver.order_items oi
JOIN silver.products p ON oi.product_id = p.product_id
GROUP BY p.category;
"""
with engine.connect() as conn:
    conn.execute(text('DROP TABLE IF EXISTS gold.category_revenue;'))
    conn.execute(text(cat_sql))
    conn.commit()

print('Category revenue (RANK + window share %):')
spark_read_pg(spark, 'gold', 'category_revenue').orderBy('revenue_rank').show(truncate=False)

In [ ]:
# ── Customer acquisition cohort ───────────────────────────────────────────────
cohort_view_sql = """
CREATE OR REPLACE VIEW gold.v_acquisition_cohort AS
SELECT
    TO_CHAR(DATE_TRUNC('month', signup_date), 'YYYY-MM') AS signup_month,
    COUNT(*)                                              AS new_customers,
    SUM(COUNT(*)) OVER (ORDER BY DATE_TRUNC('month', signup_date)) AS cumulative_customers
FROM silver.customers
WHERE signup_date IS NOT NULL
GROUP BY DATE_TRUNC('month', signup_date)
ORDER BY signup_month;
"""
with engine.connect() as conn:
    conn.execute(text(cohort_view_sql))
    conn.commit()

print('Customer acquisition cohort (SUM cumulative):')
spark_read_pg(spark, 'gold', 'v_acquisition_cohort').show(truncate=False)

---
---
## 7. Gold Summary

---

In [ ]:
from sqlalchemy import inspect as sa_inspect, text

insp = sa_inspect(engine)
tables = sorted(insp.get_table_names(schema='gold'))

print(f'{"Table":<35} {"Rows":>6}')
print('-' * 43)
for t in tables:
    with engine.connect() as conn:
        n = conn.execute(text(f'SELECT COUNT(*) FROM gold.{t}')).scalar()
    print(f'  gold.{t:<30} {n:>6}')

print('\nDay 3 complete — Gold layer ready!')

In [ ]:
spark.stop()
print('SparkSession stopped.')